# project_13_cytokine_mimetic — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:16:09 UTC
Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — cytokine signaling, target subunits, selectivity

**Standard slot:** *define & explore.* **For Project 13 this means:** understand IL-2 receptor
signaling, **choose which receptor subunits to engage** (e.g., IL-2Rβ + γc) and which to **spare**
(IL-2Rα/CD25), separate the subunits, write down the binder/selectivity metrics + cutoffs, and run a
deterministic **mock** mini-run as your "hello-world" (D0) — including a per-subunit **selectivity
profile**.

Run `00_setup.ipynb` first in this session. A real agonist campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## Why a de novo cytokine mimetic (and why selectivity)

Native **IL-2** is a powerful but toxic, unstable drug. Its receptor has three chains:
- **IL-2Rα (CD25)** — high-affinity **capture** chain; **does not signal**. Engaging it is what makes
  native IL-2 hit CD25-high regulatory T cells (Tregs) and drives vascular-leak toxicity.
- **IL-2Rβ (CD122)** + **γc (CD132)** — the **signaling pair**: dimerizing them juxtaposes JAK1/JAK3
  → **STAT5** phosphorylation → effector T/NK activation.

The **Neo-2/15** paradigm (Silva et al. 2019): a hyperstable de novo mini-protein that engages **β + γc
but has no α site** → a **βγ-biased agonist** that keeps the useful signaling, spares Tregs, and is far
more stable than IL-2. This notebook sets up that design decision and the way we will *prove* it in
silico: model each design against **each subunit separately** and report the selectivity profile.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the mimetic | thermostability / ΔG |
| **pae_interaction (per subunit)** | Å | AF2-Multimer error across the mimetic–**subunit** interface (the key metric, computed for **each** of α/β/γc) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **selectivity margin** | Å | `min(pae over spared) − max(pae over engaged)`; larger = cleaner | agonism (binding ≠ signaling) |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs (applied to the **engaged** subunits): **scRMSD ≤ 2.5, pLDDT ≥ 80,
pae_interaction ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.** `pae_interaction` is the single most important
metric — but a low value is *confidence*, **not** affinity, and a **selective binder is not an
agonist**: only a cell pSTAT5 assay decides signaling. A passing, selective design is a **hypothesis**
until per-subunit SPR **and** a STAT-phosphorylation assay.

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_13_cytokine_mimetic/notebooks


## 1 · Target prep + per-subunit hotspots

The design target is the **IL-2Rβ + γc signaling surface** (you must bridge **both** chains to dimerize
the receptor → signal), and you also keep the **IL-2Rα** subunit aside to model selectivity *against*.
Fetch the candidate complex with `data/download_data.py` (**2B5I — verify on RCSB**), identify the
chains, **split** IL-2Rα / IL-2Rβ / γc into separate targets, and read the IL-2 contact residues off
each as your per-subunit hotspots.

Below we just *declare* EXAMPLE hotspots so the notebook runs end-to-end; **replace them with the
residues you derive from the actual IL-2/IL-2R interface** (numbering depends on the PDB you verify).

In [8]:
import cytokine_tools as ct

TARGET = "IL2R_beta_gamma"           # the β+γc signaling surface (you produce this from 2B5I)
# EXAMPLE signaling-face hotspots on IL-2Rβ (chain B) and γc (chain C) — VERIFY/REPLACE from the
# IL-2/IL-2R interface (data/README.md). These are placeholders so the plumbing runs.
HOTSPOTS = ct.parse_hotspots("B41,B42,C100,C102")   # EXAMPLE_DATA placeholder residues

# The selectivity goal (Neo-2/15-style): ENGAGE the signaling pair, SPARE the capture chain.
ENGAGE = ct.ENGAGE_DEFAULT           # ("IL2Rb", "gammaC")  -> dimerize these to signal
SPARE  = ct.SPARE_DEFAULT            # ("IL2Ra",)           -> CD25, the chain we want to AVOID
print("subunits     :", ct.SUBUNITS)
print("target       :", TARGET)
print("hotspots     :", HOTSPOTS, " (EXAMPLE — replace with your verified β/γc residues)")
print("ENGAGE (signal):", ENGAGE, "  SPARE (capture/CD25):", SPARE)

subunits     : ('IL2Ra', 'IL2Rb', 'gammaC')
target       : IL2R_beta_gamma
hotspots     : ('B41', 'B42', 'C100', 'C102')  (EXAMPLE — replace with your verified β/γc residues)
ENGAGE (signal): ('IL2Rb', 'gammaC')   SPARE (capture/CD25): ('IL2Ra',)


## 2 · Mock hello-world: a tiny agonist mini-run + per-subunit selectivity

`scripts/cytokine_tools.py` exposes the design paradigms behind one API
(`generate_agonists_rfdiffusion(...)`, `generate_agonists_bindcraft(...)`), the per-subunit AF2-Multimer
scorer (`af2_multimer(seq, subunit=...)`), and the project's twist —
`selectivity_profile(...)` (model vs **each** subunit → engages βγ but not α?). The **mock** backend is
deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as real** — they
are `SYNTHETIC` by construction (and there is no fabricated EC50/K_D anywhere).

In [9]:
# A few designs, scored against EACH subunit, with a selectivity call. All numbers are SYNTHETIC.
rf = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
ct.score_designs(rf, tool="mock", engage=ENGAGE, spare=SPARE)

d = rf[0]
prof = ct.selectivity_profile(d, tool="mock", engage=ENGAGE, spare=SPARE)
print("example RFdiffusion agonist design:")
print("  id    :", d.design_id)
print("  len   :", d.length, "aa")
print("  seq   :", d.sequence)
print("  plddt :", d.plddt, " scrmsd:", d.scrmsd, " sc:", d.shape_complementarity, " (SYNTHETIC)")
print("  per-subunit pae (SELECTIVITY PROFILE):", prof["pae_by_subunit"], "(SYNTHETIC)")
print("  engaged_ok:", prof["engaged_ok"], " spared_ok:", prof["spared_ok"],
      " margin:", prof["selectivity_margin"], " -> selective:", prof["selective"])
print("  synthetic flag:", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'rfdiffusion'/'bindcraft'/'af2' on Colab (A100). See MANUAL.md §2.")

example RFdiffusion agonist design:
  id    : EXAMPLE_DATA_rfdiffusion_0000
  len   : 62 aa
  seq   : VWITGMIALMNKVMNAQDSPQMEPLDETQDIKQHYAQMYKQRNTQWSTCDEFGWYKQWSTGH
  plddt : 97.0  scrmsd: 2.67  sc: 0.72  (SYNTHETIC)
  per-subunit pae (SELECTIVITY PROFILE): {'IL2Ra': 9.0, 'IL2Rb': 9.0, 'gammaC': 6.0} (SYNTHETIC)
  engaged_ok: True  spared_ok: False  margin: 0.0  -> selective: False
  synthetic flag: True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'rfdiffusion'/'bindcraft'/'af2' on Colab (A100). See MANUAL.md §2.


## 3 · Read the selectivity profile

For a **βγ-biased** agonist you want **low** `pae` to IL-2Rβ and γc (confidently engaged → can
dimerize → signal) and **high** `pae` to IL-2Rα (the capture chain is spared). The `selective` flag
combines: engaged-OK **and** spared-OK **and** a selectivity margin above threshold. A design with low
`pae` to **all three** subunits is *not* selective — it behaves like toxic native IL-2.

In [10]:
for b in rf[:3]:
    p = ct.selectivity_profile(b, tool="mock", engage=ENGAGE, spare=SPARE)
    eng = {s: p["pae_by_subunit"][s] for s in ENGAGE}
    spr = {s: p["pae_by_subunit"][s] for s in SPARE}
    print(f"{b.design_id}: engage(low?) {eng}  spare(high?) {spr}  "
          f"margin={p['selectivity_margin']}  selective={p['selective']}  (SYNTHETIC)")
print("\nNOTE: SELECTIVE in silico is a hypothesis about which subunits are ENGAGED — not proof of")
print("agonism. Binding != signaling; the cell pSTAT5 assay (notebook 05) decides it.")

EXAMPLE_DATA_rfdiffusion_0000: engage(low?) {'IL2Rb': 9.0, 'gammaC': 6.0}  spare(high?) {'IL2Ra': 9.0}  margin=0.0  selective=False  (SYNTHETIC)
EXAMPLE_DATA_rfdiffusion_0001: engage(low?) {'IL2Rb': 12.0, 'gammaC': 16.0}  spare(high?) {'IL2Ra': 18.0}  margin=2.0  selective=False  (SYNTHETIC)
EXAMPLE_DATA_rfdiffusion_0002: engage(low?) {'IL2Rb': 2.0, 'gammaC': 7.0}  spare(high?) {'IL2Ra': 12.0}  margin=5.0  selective=False  (SYNTHETIC)

NOTE: SELECTIVE in silico is a hypothesis about which subunits are ENGAGED — not proof of
agonism. Binding != signaling; the cell pSTAT5 assay (notebook 05) decides it.


## Visualize a mimetic–receptor complex (py3Dmol)

Use this to eyeball a predicted mimetic–receptor complex once you have a real PDB (from AF2-Multimer).
For agonism, check that the mini-protein bridges **both** β and γc.

In [11]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex_beta_gamma.pdb")
print("show_complex(pdb_path) ready.")

show_complex(pdb_path) ready.


## D0 checklist
- [ ] IL-2/IL-2R accession verified on RCSB (2B5I is a candidate); chains identified (IL-2 vs α/β/γc).
- [ ] Receptor subunits **separated** (α, β, γc) + per-subunit **hotspot lists** (derived from the interface, not invented).
- [ ] **Target-subunit/selectivity choice written down** (engage β+γc, spare α — and *why*: reduce Treg/vascular-leak toxicity).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (incl. binding ≠ signaling).
- [ ] Reproduced mock mini-run with the **per-subunit selectivity profile** printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the agonist campaign engaging the chosen receptor surfaces.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — agonist mini-proteins engaging the chosen receptor surfaces

**Standard slot:** *design campaign.* **For Project 13 this is the core:** design de novo agonist
mini-proteins steered onto the **IL-2Rβ + γc signaling surface** (they must bridge both chains to
dimerize the receptor), then score **every** design against **each of the three subunits separately**
to build the selectivity data (D2):
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences (primary).
- (optional foil) **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.

Then score with **AF2-Multimer** *per subunit* (`pae_interaction` to α, β, γc — the selectivity profile).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster), and
> modeling against **three** subunits roughly **triples** the AF2-Multimer cost. Free **T4** = a *small
> fallback* (small RFdiffusion batch + ESMFold triage on β/γc; FreeBindCraft, small `num_designs`). The
> cells below run on the deterministic **mock** backend so the plumbing executes anywhere; the real
> calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [12]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_13_cytokine_mimetic/notebooks


## Version-verify the pinned upstreams (tools change!)

The design tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log it).
The generation itself needs an A100; this check needs nothing.

In [13]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # binder mode; pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   BindCraft      https://github.com/martinpacesa/BindCraft         # one-shot hallucination; pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft      # free-tier fallback — VERIFY it exists; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer (per subunit); pin <commit>
PINNED = {
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

  [200] RFdiffusion    https://github.com/RosettaCommons/RFdiffusion


  [200] ColabDesign    https://github.com/sokrypton/ColabDesign


  [200] BindCraft      https://github.com/martinpacesa/BindCraft


  [200] FreeBindCraft  https://github.com/cytokineking/FreeBindCraft


  [200] ColabFold      https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.


## 1 · Define the campaign

Same target + per-subunit hotspots as notebook 01. Set honest campaign sizes; the cells run on `mock`
so they execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the
numbers on a T4** (small RFdiffusion batch, FreeBindCraft). Remember AF2-Multimer runs **per subunit**.

In [14]:
import cytokine_tools as ct
import pandas as pd

TARGET = "IL2R_beta_gamma"
HOTSPOTS = ct.parse_hotspots("B41,B42,C100,C102")   # EXAMPLE — replace with your verified β/γc residues
ENGAGE = ct.ENGAGE_DEFAULT        # ("IL2Rb","gammaC") -> the signaling pair to engage
SPARE  = ct.SPARE_DEFAULT         # ("IL2Ra",)         -> CD25, the chain to spare
SUBUNITS = ct.SUBUNITS            # ("IL2Ra","IL2Rb","gammaC") -> AF2-Multimer is run vs EACH

# Honest campaign sizes (catalog): RFdiffusion 500-1000 backbones, BindCraft 50-200.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4

TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer, run per subunit) on Colab

print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"AF2-Multimer: tool={TOOL_AF2}  (run vs EACH subunit: {SUBUNITS})")
print("hotspots    :", HOTSPOTS, " | ENGAGE", ENGAGE, " SPARE", SPARE)

RFdiffusion : n=200 tool=mock
BindCraft   : n=60  tool=mock
AF2-Multimer: tool=mock  (run vs EACH subunit: ('IL2Ra', 'IL2Rb', 'gammaC'))
hotspots    : ('B41', 'B42', 'C100', 'C102')  | ENGAGE ('IL2Rb', 'gammaC')  SPARE ('IL2Ra',)


## 2 · Primary paradigm — RFdiffusion binder campaign → ProteinMPNN

Diffuse mini-protein backbones docked at the β/γc hotspots (supply **both** receptor chains so the
design can bridge them → dimerize → signal), then ProteinMPNN designs sequences, then AF2-Multimer
re-predicts each complex **against each subunit**. On A100 this is 500–1000 backbones (per-backbone hit
rate is low — that is normal). The `mock` backend stands in for the whole chain.

In [15]:
# Real call (Colab, A100): ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). Supply BOTH β and γc chains so the
#   mini-protein can bridge them. AF2-Multimer (×3 subunits) is the slow step. See MANUAL.md §2 / scripts.
rfdiff = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
ct.score_designs(rfdiff, tool=TOOL_AF2, engage=ENGAGE, spare=SPARE)   # fills per-subunit pae + metrics
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
ex = rfdiff[0]
print("example:", ex.design_id, "per-subunit pae =", ex.pae_by_subunit)

RFdiffusion pool: 200 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_rfdiffusion_0000 per-subunit pae = {'IL2Ra': 9.0, 'IL2Rb': 9.0, 'gammaC': 6.0}


## 3 · Optional foil — BindCraft campaign

BindCraft hallucinates a mini-protein with AF2-Multimer in the loop. Useful as a second paradigm/foil
at the same surface; we still re-score it against each subunit so the selectivity analysis is
apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [16]:
# Real call (Colab, A100): ct.generate_agonists_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/cytokine_tools.py TODOs.
bindcraft = ct.generate_agonists_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
ct.score_designs(bindcraft, tool=TOOL_AF2, engage=ENGAGE, spare=SPARE)
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
ex = bindcraft[0]
print("example:", ex.design_id, "per-subunit pae =", ex.pae_by_subunit)

BindCraft pool: 60 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_bindcraft_0000 per-subunit pae = {'IL2Ra': 11.0, 'IL2Rb': 17.0, 'gammaC': 12.0}


## 4 · Assemble + persist the pool (with per-subunit metrics)

Write one tidy CSV with the **per-subunit `pae_to_*`** columns (the selectivity data) plus the standard
binder metrics. These feed notebook 03 (the shared filter) and notebook 04 (the selectivity profile).
We add an EXAMPLE physics column (`rosetta_dG`) so the binder physics layer has something to act on in
the dry run — on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [17]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (ct._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        pae = d.pae_by_subunit or {}
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, scrmsd=d.scrmsd, shape_complementarity=d.shape_complementarity,
            pae_to_alpha=pae.get("IL2Ra"), pae_to_beta=pae.get("IL2Rb"), pae_to_gamma=pae.get("gammaC"),
            pae_interaction=d.pae_interaction,        # worst engaged-subunit pae (β/γc)
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=ct.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
combined = pd.concat([df_rf, df_bc], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined[["design_id","paradigm","pae_to_alpha","pae_to_beta","pae_to_gamma","pae_interaction"]].head(4)

wrote results/rfdiffusion_designs.csv  (200, 17)
wrote results/bindcraft_designs.csv    (60, 17)
wrote results/all_designs.csv          (260, 17)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.


,design_id,paradigm,pae_to_alpha,pae_to_beta,pae_to_gamma,pae_interaction
0,EXAMPLE_DATA_rfdiffusion_0000,rfdiffusion,9.0,9.0,6.0,9.0
1,EXAMPLE_DATA_rfdiffusion_0001,rfdiffusion,18.0,12.0,16.0,16.0
2,EXAMPLE_DATA_rfdiffusion_0002,rfdiffusion,12.0,2.0,7.0,7.0
3,EXAMPLE_DATA_rfdiffusion_0003,rfdiffusion,21.0,17.0,17.0,17.0


## D2 checklist
- [ ] RFdiffusion-binder pool generated at honest scale (500–1000 backbones → ProteinMPNN on A100).
- [ ] (optional foil) BindCraft pool generated (50–200; FreeBindCraft/small on T4).
- [ ] Every design scored by AF2-Multimer **against each of α/β/γc** (`pae_to_*` parsed); pool written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the engaged subunits.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 13** you build `fp.Design` **binder** objects from the pool, filtering on the
**engaged** signaling subunits (β, γc), call `fp.run_pipeline(..., design_type="binder")`, and
`fp.report(...)` the survival funnel + ranked CSV (D3 part 1). The **selectivity** modeling (engages βγ
but not α) is layered on top in notebook 04.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/rfdiffusion_designs.csv` (+ `bindcraft_designs.csv`) exist.

## Setup paths

In [18]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_13_cytokine_mimetic/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6. We apply `pae` to
the **worst engaged subunit** (β or γc) — a design must confidently engage **both** signaling chains.

In [19]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `Design` (binder) objects from the pool

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency — here `pae_interaction` is the **worst
engaged-subunit** pae, i.e. `max(pae_to_beta, pae_to_gamma)`), and
`rosetta_dG`/`shape_complementarity`/`solubility` (Layer 3 physics). We keep the **per-subunit** paes +
`paradigm` in `extra` for the selectivity analysis in notebook 04. (Mock has no independent orthogonal
predictor, so we run Layers 1+3 here; on Colab add a second predictor for Layer 2.)

In [20]:
import os
import pandas as pd

# Regenerate the pool if a fresh session lost it (deterministic mock).
if not os.path.exists("results/rfdiffusion_designs.csv"):
    import cytokine_tools as ct
    TARGET, HOTSPOTS = "IL2R_beta_gamma", ct.parse_hotspots("B41,B42,C100,C102")
    rf = ct.generate_agonists_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock")
    ct.score_designs(rf, tool="mock")
    rows=[]
    for d in rf:
        pae=d.pae_by_subunit or {}
        rows.append(dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                         plddt=d.plddt, scrmsd=d.scrmsd, shape_complementarity=d.shape_complementarity,
                         pae_to_alpha=pae.get("IL2Ra"), pae_to_beta=pae.get("IL2Rb"), pae_to_gamma=pae.get("gammaC"),
                         pae_interaction=d.pae_interaction,
                         rosetta_dG=round(-45.0+(ct._hashints("dG",d.design_id)%40),2), solubility=0.3,
                         hotspot_overlap=ct.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic))
    pd.DataFrame(rows).to_csv("results/rfdiffusion_designs.csv", index=False)

# Combine whichever pools exist (RFdiffusion primary; BindCraft foil if present).
frames = [pd.read_csv("results/rfdiffusion_designs.csv")]
if os.path.exists("results/bindcraft_designs.csv"):
    frames.append(pd.read_csv("results/bindcraft_designs.csv"))
pool = pd.concat(frames, ignore_index=True)

def worst_engaged_pae(r):
    vals = [r.get("pae_to_beta"), r.get("pae_to_gamma")]
    vals = [v for v in vals if pd.notna(v)]
    return max(vals) if vals else r.get("pae_interaction")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=worst_engaged_pae(r), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"),
               "pae_to_alpha": r.get("pae_to_alpha"), "pae_to_beta": r.get("pae_to_beta"),
               "pae_to_gamma": r.get("pae_to_gamma"), "hotspot_overlap": r.get("hotspot_overlap")},
    )

binders = [row_to_binder(r) for _, r in pool.iterrows()]
print(f"built {len(binders)} binder Designs (pae_interaction = worst engaged subunit, β/γc)")

built 260 binder Designs (pae_interaction = worst engaged subunit, β/γc)


## Run the pipeline (engaged-subunit cutoffs)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked DataFrame
with survival counts in `df.attrs`. Filtering here is on the **engaged** signaling chains (a design must
confidently engage both β and γc). We use Layers 1+3 (mock has no independent orthogonal source; add
Layer 2 on Colab with a second predictor). **Selectivity** (sparing α) is enforced in notebook 04.

In [21]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

df_ranked = fp.run_pipeline(binders, design_type="binder", use_layers=(1, 3))
# carry the per-subunit paes + paradigm out of extra for downstream use
for col in ["paradigm", "pae_to_alpha", "pae_to_beta", "pae_to_gamma", "hotspot_overlap"]:
    df_ranked[col] = df_ranked["extra"].apply(lambda e: e.get(col) if isinstance(e, dict) else None)

surv = df_ranked.attrs["survival"]; n = df_ranked.attrs["n_total"]
passed = int((df_ranked["layers_passed"] >= 3).sum())
print(f"pool: {n} designs, survival {surv}, all-layers (engaged) hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")

df_ranked.to_csv("results/all_ranked.csv", index=False)
print("wrote results/all_ranked.csv", df_ranked.shape)
df_ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                    "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

pool: 260 designs, survival {'L1': 40, 'L3': 17}, all-layers (engaged) hit rate = 17/260 (6.5%)
wrote results/all_ranked.csv (260, 25)


,design_id,paradigm,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG
0,EXAMPLE_DATA_rfdiffusion_0120,rfdiffusion,3,4.3533,0.97,87.0,5.0,-31.0
1,EXAMPLE_DATA_rfdiffusion_0075,rfdiffusion,3,4.3067,1.11,91.0,7.0,-42.0
2,EXAMPLE_DATA_rfdiffusion_0038,rfdiffusion,3,4.2267,1.27,97.0,7.0,-43.0
3,EXAMPLE_DATA_bindcraft_0017,bindcraft,3,4.1833,1.16,86.0,8.0,-45.0
4,EXAMPLE_DATA_rfdiffusion_0181,rfdiffusion,3,4.0167,1.13,93.0,10.0,-40.0
5,EXAMPLE_DATA_rfdiffusion_0118,rfdiffusion,3,3.9867,1.15,85.0,7.0,-31.0
6,EXAMPLE_DATA_rfdiffusion_0061,rfdiffusion,3,3.7967,1.32,82.0,10.0,-44.0
7,EXAMPLE_DATA_bindcraft_0008,bindcraft,3,3.7367,1.30,90.0,10.0,-36.0
8,EXAMPLE_DATA_rfdiffusion_0002,rfdiffusion,3,3.5267,1.72,82.0,7.0,-38.0
9,EXAMPLE_DATA_rfdiffusion_0050,rfdiffusion,3,3.3000,1.96,96.0,9.0,-40.0


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Read the bars as a funnel:
steep drops show which layer discriminates. (This counts designs that confidently **engage** β/γc; the
selectivity filter in notebook 04 then asks which of these also **spare** α.)

In [22]:
top = fp.report(df_ranked, top_n=15, save_prefix="results/p13")
print("\nsaved results/p13_survival.png + results/p13_ranked.csv")
top

Total designs: 260
  L1 survivors: 40  (15.4%)
  L3 survivors: 17  (6.5%)



saved results/p13_survival.png + results/p13_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_rfdiffusion_0120,binder,3,4.3533,0.97,87.0,5.0,-31.0,None
1,EXAMPLE_DATA_rfdiffusion_0075,binder,3,4.3067,1.11,91.0,7.0,-42.0,None
2,EXAMPLE_DATA_rfdiffusion_0038,binder,3,4.2267,1.27,97.0,7.0,-43.0,None
3,EXAMPLE_DATA_bindcraft_0017,binder,3,4.1833,1.16,86.0,8.0,-45.0,None
4,EXAMPLE_DATA_rfdiffusion_0181,binder,3,4.0167,1.13,93.0,10.0,-40.0,None
5,EXAMPLE_DATA_rfdiffusion_0118,binder,3,3.9867,1.15,85.0,7.0,-31.0,None
6,EXAMPLE_DATA_rfdiffusion_0061,binder,3,3.7967,1.32,82.0,10.0,-44.0,None
7,EXAMPLE_DATA_bindcraft_0008,binder,3,3.7367,1.30,90.0,10.0,-36.0,None
8,EXAMPLE_DATA_rfdiffusion_0002,binder,3,3.5267,1.72,82.0,7.0,-38.0,None
9,EXAMPLE_DATA_rfdiffusion_0050,binder,3,3.3000,1.96,96.0,9.0,-40.0,None


## Honest hit-rate accounting

Report `N passing all (engaged) layers / N generated`. This is the **engagement** hit rate; notebook 04
adds the **selectivity** hit rate (how many engaged survivors also spare α). Remember: survival is
*enrichment*, not *correctness*, and engaging β/γc is not yet agonism. Mock numbers are SYNTHETIC.

In [23]:
dist = df_ranked["layers_passed"].value_counts().sort_index().to_dict()
n = len(df_ranked); passed = int((df_ranked["layers_passed"] >= 3).sum())
print("layers_passed distribution:", dist)
print(f"engagement hit rate (pass all engaged layers): {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")
if "paradigm" in df_ranked.columns:
    for p, g in df_ranked.groupby("paradigm"):
        gp = int((g["layers_passed"] >= 3).sum())
        print(f"  {p:12s}: {gp}/{len(g)} ({100*gp/max(len(g),1):.1f}%)")

layers_passed distribution: {0: 220, 1: 23, 3: 17}
engagement hit rate (pass all engaged layers): 17/260 (6.5%)  [SYNTHETIC if mock]
  bindcraft   : 2/60 (3.3%)
  rfdiffusion : 15/200 (7.5%)


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Filtering applied to the **engaged** subunits (pae = worst of β/γc); per-subunit paes carried through.
- [ ] Survival-at-each-layer reported (funnel figure `results/p13_survival.png`).
- [ ] Honest **engagement** hit-rate accounting (N pass / N generated).
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the subunit-selectivity profile + stability-vs-native analysis.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — subunit-selectivity profile + stability vs native cytokine

**Standard slot:** *validate (in silico).* **For Project 13 this is the core analysis:** turn the
per-subunit `pae` into a **selectivity profile** (engages IL-2Rβ + γc but **not** IL-2Rα), quantify the
**selectivity margin**, and compare the de novo mimetics' **stability to the native cytokine** (the
Neo-2/15 selling point), with publication-style figures (D3 part 2).

Needs `results/all_ranked.csv` (notebook 03) and the design pool CSV(s) (notebook 02).

## Setup paths

In [24]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_13_cytokine_mimetic/notebooks


## 1 · Build the selectivity profile

For each design we have `pae_to_alpha / beta / gamma`. A **βγ-biased** agonist (Neo-2/15-style) should
have **low** `pae` to β and γc (engaged signaling pair) and **high** `pae` to α (spared capture chain).
We compute, per design: `engaged_ok` (β and γc ≤ 10), `spared_ok` (α ≥ 14), and the **selectivity
margin** `= pae_to_alpha − max(pae_to_beta, pae_to_gamma)` (larger ⇒ cleaner βγ bias). Mock numbers are
SYNTHETIC; the thresholds are project-specific — justify yours.

In [25]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
ENGAGE_MAX_PAE = 10.0   # β and γc must be <= this to count as ENGAGED (matches the binder cutoff)
SPARE_MIN_PAE  = 14.0   # α must be >= this to count as SPARED (project-specific; justify it)
MARGIN_MIN     = 4.0    # selectivity margin threshold (project-specific)

def selectivity_row(r):
    a, b, g = r.get("pae_to_alpha"), r.get("pae_to_beta"), r.get("pae_to_gamma")
    if pd.isna(a) or pd.isna(b) or pd.isna(g):
        return pd.Series(dict(engaged_ok=False, spared_ok=False, sel_margin=np.nan, selective=False))
    engaged_ok = (b <= ENGAGE_MAX_PAE) and (g <= ENGAGE_MAX_PAE)
    spared_ok  = (a >= SPARE_MIN_PAE)
    margin = a - max(b, g)
    selective = bool(engaged_ok and spared_ok and margin >= MARGIN_MIN)
    return pd.Series(dict(engaged_ok=engaged_ok, spared_ok=spared_ok,
                          sel_margin=round(float(margin), 2), selective=selective))

ranked = pd.concat([ranked, ranked.apply(selectivity_row, axis=1)], axis=1)

n = len(ranked)
n_eng_pass = int((ranked["layers_passed"] >= 3).sum())             # engaged the filter (nb 03)
n_selective = int(ranked["selective"].sum())                       # selective by profile
n_sel_and_pass = int((ranked["selective"] & (ranked["layers_passed"] >= 3)).sum())  # the real target
print(f"pool                              : {n}")
print(f"pass engaged filter (nb03)        : {n_eng_pass} ({100*n_eng_pass/max(n,1):.1f}%)")
print(f"selective by profile (βγ, not α)  : {n_selective} ({100*n_selective/max(n,1):.1f}%)")
print(f"SELECTIVE AGONIST CANDIDATES      : {n_sel_and_pass}  (pass filter AND selective)  [SYNTHETIC if mock]")
ranked.to_csv("results/all_ranked_selectivity.csv", index=False)
print("wrote results/all_ranked_selectivity.csv")

pool                              : 260
pass engaged filter (nb03)        : 17 (6.5%)
selective by profile (βγ, not α)  : 56 (21.5%)
SELECTIVE AGONIST CANDIDATES      : 9  (pass filter AND selective)  [SYNTHETIC if mock]
wrote results/all_ranked_selectivity.csv


## 2 · Plot the selectivity profile

The clean separation we want: selective designs sit at **low `pae` to β/γc** and **high `pae` to α**.
We plot engaged-pae (worst of β/γc) vs α-pae; selective agonist candidates fall in the upper-left
(engaged & spared). The dashed lines are the thresholds.

In [26]:
eng_pae = ranked[["pae_to_beta", "pae_to_gamma"]].max(axis=1)   # worst engaged subunit
a_pae = ranked["pae_to_alpha"]
sel = ranked["selective"].fillna(False).astype(bool)

fig, ax = plt.subplots(figsize=(5.4, 4.2))
ax.scatter(eng_pae[~sel], a_pae[~sel], s=16, alpha=0.5, label="not selective", color="#888888")
ax.scatter(eng_pae[sel],  a_pae[sel],  s=22, alpha=0.85, label="selective (βγ, not α)", color="#1f77b4")
ax.axvline(ENGAGE_MAX_PAE, ls="--", c="green", lw=1, label=f"engage ≤ {ENGAGE_MAX_PAE}")
ax.axhline(SPARE_MIN_PAE,  ls="--", c="red",   lw=1, label=f"spare α ≥ {SPARE_MIN_PAE}")
ax.set_xlabel("pae to engaged subunit (max of β, γc) — lower = engaged")
ax.set_ylabel("pae to IL-2Rα (CD25) — higher = spared")
ax.set_title("Subunit-selectivity profile (EXAMPLE_DATA if mock)")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig("results/p13_selectivity.png", dpi=150); plt.show()
print("saved results/p13_selectivity.png")

saved results/p13_selectivity.png


## 3 · Selectivity margin distribution `[extension]`

The **selectivity margin** (`pae_to_α − max(pae_to_β, pae_to_γ)`) summarizes the profile in one number.
A right-shifted distribution means cleaner βγ bias. Compare paradigms if you ran both.

In [27]:
fig, ax = plt.subplots(figsize=(5.6, 3.4))
if "paradigm" in ranked.columns and ranked["paradigm"].notna().any():
    for p, gdf in ranked.groupby("paradigm"):
        ax.hist(gdf["sel_margin"].dropna(), bins=15, alpha=0.5, label=str(p))
    ax.legend(fontsize=8)
else:
    ax.hist(ranked["sel_margin"].dropna(), bins=15, alpha=0.7)
ax.axvline(MARGIN_MIN, ls="--", c="k", lw=1, label=f"margin ≥ {MARGIN_MIN}")
ax.set_xlabel("selectivity margin (Å):  pae_to_α − max(pae_to_β, pae_to_γ)")
ax.set_ylabel("designs"); ax.set_title("Selectivity margin (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p13_margin.png", dpi=150); plt.show()
print("median selectivity margin:", round(float(ranked["sel_margin"].median(skipna=True)), 2), "(SYNTHETIC if mock)")
print("saved results/p13_margin.png")

median selectivity margin: 4.0 (SYNTHETIC if mock)
saved results/p13_margin.png


## 4 · Stability vs the native cytokine `[extension]`

The Neo-2/15 selling point: de novo mimetics can be **far more thermostable** than native IL-2 (no
disulfide dependence, idealized core). Here we **scaffold** the comparison with a clearly-labeled
EXAMPLE stability proxy — **on Colab replace it with a real proxy** (e.g., a short MD melting-RMSF, a
ΔΔG/FoldX-style score, or in the wet lab a **DSF Tm**). We anchor an EXAMPLE native-IL-2 reference value
purely so the bar chart renders; **it is not a measurement**.

In [28]:
import cytokine_tools as ct

# EXAMPLE stability proxy (SYNTHETIC): a deterministic per-design "stability score" stand-in. On Colab,
# replace with a real proxy (MD RMSF / ΔΔG) or DSF Tm in the wet lab. NOT a measurement.
def example_stability_proxy(design_id):
    return round(45.0 + (ct._hashints("stab", design_id) % 350) / 10.0, 1)   # ~45-80 (arbitrary units)

NATIVE_IL2_PROXY = 55.0   # EXAMPLE_DATA anchor for native IL-2 on the SAME arbitrary scale (NOT a real Tm)

ranked["stability_proxy"] = ranked["design_id"].map(example_stability_proxy)
selective = ranked[ranked["selective"].fillna(False)]
med_design = float(ranked["stability_proxy"].median())
med_sel = float(selective["stability_proxy"].median()) if len(selective) else float("nan")

fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.bar(["native IL-2\n(EXAMPLE ref)", "all designs\n(median)", "selective\n(median)"],
       [NATIVE_IL2_PROXY, med_design, med_sel], color=["#bbbbbb", "#9ecae1", "#1f77b4"])
ax.set_ylabel("stability proxy (arbitrary, EXAMPLE_DATA)")
ax.set_title("Stability vs native cytokine (EXAMPLE_DATA — replace with MD/ΔΔG or DSF Tm)")
plt.tight_layout(); plt.savefig("results/p13_stability.png", dpi=150); plt.show()
print(f"median stability proxy — all designs: {med_design}, selective: {med_sel}, native ref: {NATIVE_IL2_PROXY}")
print("ALL stability numbers here are EXAMPLE_DATA/SYNTHETIC — replace with a real proxy/DSF and never report as measured.")
print("saved results/p13_stability.png")

median stability proxy — all designs: 63.849999999999994, selective: 64.5, native ref: 55.0
ALL stability numbers here are EXAMPLE_DATA/SYNTHETIC — replace with a real proxy/DSF and never report as measured.
saved results/p13_stability.png


## 5 · Select the top selective-agonist candidates

The D★ deliverable wants **subunit-selective agonist designs** carried forward. Rank the
filter-passing **and** selective designs by the composite score, tie-broken by selectivity margin, and
save the shortlist for the validation plan (notebook 05).

In [29]:
cand = ranked[(ranked["layers_passed"] >= 3) & (ranked["selective"].fillna(False))].copy()
cand = cand.sort_values(["score", "sel_margin"], ascending=False).head(20)
cand.to_csv("results/top_candidates.csv", index=False)
print(f"wrote results/top_candidates.csv: {cand.shape} (filter-passing AND selective; top<=20)")
if "paradigm" in cand.columns:
    print("by paradigm:", cand.groupby("paradigm").size().to_dict())
cand.head(8)[["design_id", "paradigm", "score", "pae_to_alpha", "pae_to_beta", "pae_to_gamma", "sel_margin"]]

wrote results/top_candidates.csv: (9, 30) (filter-passing AND selective; top<=20)
by paradigm: {'bindcraft': 1, 'rfdiffusion': 8}


,design_id,paradigm,score,pae_to_alpha,pae_to_beta,pae_to_gamma,sel_margin
0,EXAMPLE_DATA_rfdiffusion_0120,rfdiffusion,4.3533,19.0,5.0,3.0,14.0
1,EXAMPLE_DATA_rfdiffusion_0075,rfdiffusion,4.3067,21.0,7.0,2.0,14.0
2,EXAMPLE_DATA_rfdiffusion_0038,rfdiffusion,4.2267,22.0,5.0,7.0,15.0
4,EXAMPLE_DATA_rfdiffusion_0181,rfdiffusion,4.0167,14.0,3.0,10.0,4.0
5,EXAMPLE_DATA_rfdiffusion_0118,rfdiffusion,3.9867,16.0,7.0,7.0,9.0
7,EXAMPLE_DATA_bindcraft_0008,bindcraft,3.7367,15.0,10.0,7.0,5.0
10,EXAMPLE_DATA_rfdiffusion_0065,rfdiffusion,3.2333,23.0,5.0,5.0,18.0
11,EXAMPLE_DATA_rfdiffusion_0034,rfdiffusion,2.9533,24.0,4.0,5.0,19.0


## D3 (part 2) checklist
- [ ] **Per-subunit selectivity profile** built (engages β+γc, spares α) with justified thresholds.
- [ ] Selectivity figure (`results/p13_selectivity.png`) + selectivity-margin distribution (`results/p13_margin.png`).
- [ ] **Stability-vs-native** comparison (`results/p13_stability.png`) — with a real proxy on Colab, EXAMPLE_DATA flagged here.
- [ ] Two honest numbers: engagement hit rate (nb03) **and** selective-agonist-candidate count (selective ∧ pass).
- [ ] `results/top_candidates.csv`: selective agonist candidates, ready for the validation plan.
- [ ] Stated plainly: selective in silico ≠ an agonist; binding ≠ signaling.

**Next:** `05_validation_plan.ipynb` — per-subunit SPR + the cell STAT-phosphorylation assay plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — per-subunit SPR + cell STAT-phosphorylation + controls

**Standard slot:** *validation plan.* **For Project 13 this means:** turn the selective candidates into
a **costed, controlled wet-lab plan** — **per-subunit SPR/BLI** (K_D to IL-2Rα, IL-2Rβ, γc *separately*
→ confirm selectivity), **a cell-based STAT-phosphorylation (pSTAT5) assay** (the readout that decides
agonism, because **binding ≠ signaling**), the mandatory controls (positive: native IL-2 / Neo-2/15;
**scrambled-interface** negative; unrelated negative), an expression strategy, a **thermostability**
comparison to native IL-2, and the **Boltz-2 affinity** stretch (scaffold only) (D4/D5).

A design that passes every filter and looks selective is a **hypothesis** — per-subunit SPR + pSTAT5 is
what tests it. Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [30]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_13_cytokine_mimetic/notebooks


## 1 · Draft the experimental validation plan

Generate a plan card from the top selective candidates: per-subunit assays, the signaling readout,
controls, expression, timeline, costed reagents. Fill the `<...>` from your own numbers; this is the
deliverable other people will actually read.

In [31]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if (n_top and "paradigm" in top.columns) else {}

plan = f"""# IL-2 Cytokine-Mimetic Validation Plan (Project 13 — by <your name>, <date>)

## Candidates
Top {n_top} SELECTIVE-AGONIST candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, not affinity;
SELECTIVE in silico is not an agonist — BINDING IS NOT SIGNALING. Do NOT fabricate an EC50/K_D.

## Expression strategy
- Mimetic: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa), de novo ->
  high yield + high stability expected (the Neo-2/15 advantage).
- Receptor-subunit ectodomain reagents (IL-2Ra/CD25, IL-2Rb/CD122, gammaC/CD132): mammalian/insect
  expression or commercial; confirm each is active before testing.

## Assays (go/no-go -> selectivity -> signaling)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse?).
2. SELECTIVITY (per-subunit SPR/BLI): measure K_D + kinetics to IL-2Ra, IL-2Rb, AND gammaC SEPARATELY.
   Expect: binds IL-2Rb and gammaC; does NOT (or weakly) bind IL-2Ra -> confirms the βγ-bias in vitro.
3. SIGNALING (the decisive readout): cell-based STAT-phosphorylation (pSTAT5) assay on IL-2-responsive
   cells (e.g., CTLL-2 or primary T/NK). Dose-response -> EC50, Emax (full vs partial agonist?).
   Run on CD25+ vs CD25- cells to confirm alpha-INDEPENDENT signaling (the selectivity payoff:
   activates effector cells, spares CD25-high Tregs).
4. Stability: DSF (Tm) vs native IL-2 -> quantify the thermostability advantage.
   Deep (optional): co-crystal / cryo-EM of the mimetic-receptor complex; in-vivo Treg-vs-effector.

## Controls (MANDATORY)
- Positive: native IL-2 (and/or Neo-2/15) -> confirms SPR reagents + the pSTAT5 assay/cells respond.
- Negative (scrambled-interface): YOUR OWN top design with its beta/gammaC interface residues
  scrambled/mutated -> must LOSE binding AND signaling (cleanest specificity control).
- Negative (unrelated): an unrelated mini-protein of similar size -> should not bind or signal.

## Realistic expectations
De novo agonist design with clean subunit selectivity is HARD. In-silico hit rates vary widely; the
MAJORITY of in-silico hits fail experimentally, and even an experimental BINDER may fail to SIGNAL
(wrong dimerizing geometry). Report the experimental hit rate honestly. A selective binder that does
NOT trigger pSTAT5 is a negative result worth reporting.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} mimetics + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- Receptor-subunit reagents (a/b/gammaC) + SPR/BLI chips + native IL-2 positive control: $<...>.
- pSTAT5 assay (antibodies, IL-2-responsive cells, CD25+/- lines, flow time): $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
A receptor-selective agonist mimicking a human cytokine for cancer immunotherapy / immune modulation,
whose explicit goal is to REDUCE the toxicity of native IL-2 (βγ-biased -> spares CD25-high Tregs and
vascular-leak toxicity) -> in scope, LOW dual-use risk. Gene synthesis via a biosecurity-screening
provider; cell-based immune assays under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

wrote results/validation_plan.md — fill the <...> placeholders from your numbers.
# IL-2 Cytokine-Mimetic Validation Plan (Project 13 — by <your name>, <date>)

## Candidates
Top 9 SELECTIVE-AGONIST candidates carried forward ({'bindcraft': 1, 'rfdiffusion': 8}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. pae_interaction is confidence, not affinity;
SELECTIVE in silico is not an agonist — BINDING IS NOT SIGNALING. Do NOT fabricate an EC50/K_D.

## Expression strategy
- Mimetic: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa), de novo ->
  high yield + high stability expected (the Neo-2/15 advantage).
- Receptor-subunit ectodomain reagents (IL-2Ra/CD25, IL-2Rb/CD122, gammaC/CD132): mammalian/insect
 ...


## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its β/γc interface
residues** (the positions contacting the signaling chains) — it should **lose** binding **and**
signaling. Generating these alongside the real designs (same expression batch) makes the SPR + pSTAT5
comparison airtight. Here we scaffold the sequence-level scramble deterministically; on Colab, scramble
the predicted **interface** positions specifically using the predicted contacts.

In [32]:
import random
import cytokine_tools as ct   # ct._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting β/γc)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s:
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=ct._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control (must lose binding AND signaling)"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

wrote results/negative_controls.csv: 9 scrambled-interface negatives


## 3 · (Stretch) Boltz-2 affinity per subunit on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes — run it **per subunit** to
corroborate the selectivity ranking. Use it for **relative ranking + caveats only** — **never fabricate
a K_D/EC50**, and never present a predicted number as measured. This tells you which hits to test
first, not whether they bind or signal.

In [33]:
# Scaffold ONLY. Do NOT invent affinities. On Colab:
#   pip install boltz; build each (mimetic, subunit) complex input for α/β/γc; run boltz predict with
#   affinity mode; read the predicted-affinity signal PER SUBUNIT and report the RELATIVE ranking of the
#   top hits + heavy caveats (corroborates the AF2 selectivity profile).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking per subunit + caveats only, NEVER a fabricated K_D/EC50.")
print("Use it to PRIORITIZE which selective hits to test first in SPR/pSTAT5 — not as evidence of binding or signaling.")

Boltz-2 affinity is a STRETCH scaffold: relative ranking per subunit + caveats only, NEVER a fabricated K_D/EC50.
Use it to PRIORITIZE which selective hits to test first in SPR/pSTAT5 — not as evidence of binding or signaling.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **per-subunit SPR/BLI** + the **cell pSTAT5 signaling** assay, expression, timeline, costed reagents.
- [ ] Controls specified: positive (native IL-2 / Neo-2/15), **scrambled-interface** negative (`results/negative_controls.csv`), unrelated negative.
- [ ] Selectivity confirmed in vitro (SPR to α/β/γc separately) AND agonism tested (pSTAT5, CD25± cells).
- [ ] Thermostability vs native IL-2 planned (DSF Tm); (stretch) Boltz-2 per-subunit ranking only — no fabricated K_D/EC50.
- [ ] Honest framing: every design is a hypothesis; selective in silico ≠ agonist; binding ≠ signaling; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a subunit-selective agonist design + selectivity profile + a controlled per-subunit-SPR +
STAT-signaling validation plan, following the binder-family template with a selectivity twist.